# Docling smoke test — Aleem textbook preprocessing

**Goal:** validate that Docling extracts digital text + structure (headings, tables) from a Saudi textbook PDF, with correct Arabic RTL handling, and *without* OCR.

Upload one representative PDF (ideally a chapter or two, not the whole book) and run cells top to bottom.

Checks:
1. Markdown export from text layer (no OCR)
2. Heading hierarchy — sanity check for lesson-level chunking
3. Table extraction count
4. Arabic sample — visual check that text is in logical reading order
5. Optional OCR comparison if Path A looks broken

In [ ]:
!pip install -q docling

In [ ]:
import re, time, textwrap
from pathlib import Path

from google.colab import files
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

In [ ]:
uploaded = files.upload()
pdf_path = Path(next(iter(uploaded)))
print(f"Uploaded: {pdf_path}  ({pdf_path.stat().st_size / 1024:.0f} KB)")

## Path A — text-layer extraction (no OCR)

Digital PDFs already contain the text. This pulls it directly via Docling's layout pipeline — fast and free of OCR errors.

In [ ]:
pipeline_options = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True,
)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

t0 = time.time()
result = converter.convert(str(pdf_path))
elapsed = time.time() - t0
print(f"Converted in {elapsed:.1f}s")

In [ ]:
md = result.document.export_to_markdown()
out_path = pdf_path.with_suffix(".md")
out_path.write_text(md, encoding="utf-8")
print(f"Wrote {out_path} ({len(md):,} chars)")
print("\n--- First 1500 chars ---\n")
print(md[:1500])

In [ ]:
# Heading hierarchy — does Docling promote textbook lesson titles to H1/H2/H3?
heading_counts = {}
headings = []
for line in md.splitlines():
    m = re.match(r"^(#{1,6})\s", line)
    if m:
        level = len(m.group(1))
        heading_counts[level] = heading_counts.get(level, 0) + 1
        headings.append(line)

print("Heading counts by level:")
for lvl in sorted(heading_counts):
    print(f"  H{lvl}: {heading_counts[lvl]}")

print("\n--- All detected headings ---")
for h in headings:
    print(h)

In [ ]:
# Table count (rough — counts markdown table separator rows)
table_sep_rows = sum(1 for line in md.splitlines() if re.match(r"^\|\s*-{3,}", line))
print(f"Markdown table separator rows: {table_sep_rows}")

# Arabic sample — pull the first paragraph containing Arabic glyphs
arabic_re = re.compile(r"[\u0600-\u06FF]")
for para in md.split("\n\n"):
    if arabic_re.search(para):
        print("\n--- First Arabic paragraph ---")
        print(textwrap.shorten(para.strip(), width=600, placeholder=" ..."))
        break
else:
    print("No Arabic characters detected in markdown output.")

In [ ]:
files.download(str(out_path))

## Path B — OCR comparison (optional)

Run only if Path A is missing chunks, the Arabic looks garbled, or headings aren't being detected. This forces OCR on every page using EasyOCR (Docling's bundled default; no extra install).

In [ ]:
from docling.datamodel.pipeline_options import EasyOcrOptions

pipeline_options_ocr = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=EasyOcrOptions(lang=["ar", "en"]),
)
converter_ocr = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options_ocr)}
)

t0 = time.time()
result_ocr = converter_ocr.convert(str(pdf_path))
print(f"OCR conversion took {time.time() - t0:.1f}s")

md_ocr = result_ocr.document.export_to_markdown()
out_path_ocr = pdf_path.with_name(pdf_path.stem + "_ocr.md")
out_path_ocr.write_text(md_ocr, encoding="utf-8")
print(f"Wrote {out_path_ocr} ({len(md_ocr):,} chars)")
print("\n--- First 1500 chars ---\n")
print(md_ocr[:1500])
files.download(str(out_path_ocr))

## What to look for in the output

- **Headings detected:** if H1/H2/H3 counts are near zero, Docling isn't recognizing lesson titles — chunking-by-topic will need a different signal (font size, regex on lesson-marker words, etc.).
- **Arabic reading order:** copy a line of Arabic from the source PDF, search for it in the markdown output. The first/last word should match (not reversed).
- **Tables:** if the book has tables and `table_sep_rows == 0`, table reconstruction failed.
- **Path A vs B:** if A is empty/garbled but B looks right, the PDF text layer is broken and you'll need OCR despite the digital appearance. If A is fine, ship A.